# **Rent Analysis in Seoul**

## **Introduction**

The aim of this analysis is to find the key about why the rentals in South Korea's rental market have been increasing rapidly in recent years, and to identify the factors that contribute to this trend.

This analysis will focus in datasets extracted form Seoul Open Data Plaza (https://data.seoul.go.kr/#), and will include data on rental prices, housing types, and other relevant factors.

The datasets used in this analysis are:
- Seoul Real Estate Jeonse and Monthly Rent Price Information (https://data.seoul.go.kr/dataList/OA-21276/S/1/datasetView.do)

I take the datasets from the years 2022, 2023, 2024 and 2025 , which are the most recent years available in the dataset. All the datasets where merged into one dataset for the analysis and take randomly 400000 rows for the analysis.

The information about the columns from the dataset are the next:
<center>

|Column Name|Column Description|Notes|
|:---|:-------|:-------|
|`registration_year`|Year the contract was registered (file year).|Not the contract year; use `contract_date` for that. A few rows are 2021.|
|`district_code`|Code of the Seoul district (gu).|5 digits. See Lookup_Dongs.|
|`district_name`|District name in English.|Recovered from `district_code` (25 districts).|
|`legal_dong_code`|Code of the legal neighborhood (dong) within the district.|Only unique together with `district_code`.|
|`legal_dong_name`|Neighborhood name, romanized.|Automatic romanization; Korean original in Lookup_Dongs. Blank for a few unmapped codes.|
|`lot_type_code`|Type of lot number: 1, 2 or 3.|Blank for detached / multi-household houses (no lot number in source).|
|`lot_type`|Land lot / Mountain lot / Block lot.|From `lot_type_code`.|
|`main_lot_number`|Main lot number (jibun) of the address.||
|`sub_lot_number`|Sub lot number (jibun) of the address.|0 = none.|
|`floor`|Floor of the unit.|Negative = basement. Blank if not reported.|
|`contract_date`|Date the contract was signed.|Format yyyy-mm-dd.|
|`lease_type`|Jeonse (deposit only) or Monthly rent.|2023: original value. Other years: inferred (monthly rent = 0 -> Jeonse); matches the original in 99.96% of 2023 rows.|
|`leased_area_sqm`|Leased area in square meters (m2).||
|`deposit_10k_krw`|Deposit, in 10,000 KRW.|Example: 16000 = 160,000,000 KRW.|
|`monthly_rent_10k_krw`|Monthly rent, in 10,000 KRW.|0 for Jeonse.|
|`building_name`|Building name (Korean).|Only available for 2023 rows; unrecoverable in 2022/2024/2025 files. Blank when unnamed.|
|`year_built`|Year the building was built.||
|`building_type`|Apartment / Officetel / Row house-Villa / Detached-Multi-household house.|Recovered in all years.|
|`contract_period`|Lease term as YY.MM~YY.MM.|Example: 22.01~24.01 = Jan 2022 to Jan 2024.|
|`contract_type`|New contract or Renewal.|Blank = not specified in source.|
|`renewal_right_used`|Yes if the tenant used the renewal-request right.|Blank = no / not reported.|
|`previous_deposit_10k_krw`|Deposit of the previous contract (renewals), in 10,000 KRW.||
|`previous_monthly_rent_10k_krw`|Monthly rent of the previous contract (renewals), in 10,000 KRW.||

</center>

## **1: Imports** 

These are the libraries used in this analysis:

In [2]:
import pandas as pd

## **2: Read dataset** 

The method **.read_csv** from the library **pandas** reads a CSV file and returns a DataFrame. It is used to load the dataset into a pandas DataFrame for further analysis.

The syntax is:
pd.read_csv('file_path')

In this analysis, the dataset is loaded from the file path `'C:\\Users\\jamol\\Datasets\\Seoul_Rentals_2022_2025.csv'` and stored in the variable `df`.

In [3]:
df = pd.read_csv('C:\\Users\\jamol\\Datasets\\Seoul_Rentals_2022_2025.csv')

## **3: Data exploration**

In this section, we will explore the dataset to understand its structure and contents. We will start by displaying the first few rows of the dataset using the `head()` method. This will give us a glimpse of the data and help us identify any potential issues or areas for further analysis.

In [4]:
df.head(10)

,registration_year,district_code,district_name,legal_dong_code,legal_dong_name,lot_type_code,lot_type,main_lot_number,sub_lot_number,floor,...,deposit_10k_krw,monthly_rent_10k_krw,building_name,year_built,building_type,contract_period,contract_type,renewal_right_used,previous_deposit_10k_krw,previous_monthly_rent_10k_krw
0,2022,11110,Jongno-gu,17500,Sungin-dong,1.0,Land lot,204.0,11.0,8.0,...,24000,0,NaN,1979.0,Apartment,22.01~23.12,Renewal,Yes,23000.0,NaN
1,2022,11170,Yongsan-gu,10200,Yongsan-dong 2-ga,NaN,NaN,NaN,NaN,NaN,...,500,40,NaN,1985.0,Detached / Multi-household house,22.01~24.01,New,NaN,0.0,0.0
2,2022,11170,Yongsan-gu,11100,Cheongpa-dong 3-ga,NaN,NaN,NaN,NaN,NaN,...,1000,40,NaN,1970.0,Detached / Multi-household house,22.01~23.12,New,NaN,0.0,0.0
3,2022,11170,Yongsan-gu,12200,Munbae-dong,1.0,Land lot,24.0,4.0,13.0,...,17500,10,NaN,2013.0,Officetel,22.01~22.12,Renewal,NaN,17500.0,0.0
4,2022,11200,Seongdong-gu,10800,Eungbong-dong,1.0,Land lot,265.0,24.0,1.0,...,11000,0,NaN,1992.0,Row house / Villa (multi-unit),NaN,NaN,NaN,NaN,NaN
5,2022,11215,Gwangjin-gu,10100,Junggok-dong,1.0,Land lot,150.0,323.0,1.0,...,10000,60,NaN,1997.0,Row house / Villa (multi-unit),22.02~24.02,New,NaN,0.0,0.0
6,2022,11215,Gwangjin-gu,10100,Junggok-dong,NaN,NaN,NaN,NaN,NaN,...,500,30,NaN,1997.0,Detached / Multi-household house,NaN,NaN,NaN,NaN,NaN
7,2022,11215,Gwangjin-gu,10700,Hwayang-dong,1.0,Land lot,111.0,12.0,8.0,...,1000,80,NaN,2021.0,Officetel,22.01~23.01,New,NaN,0.0,0.0
8,2022,11215,Gwangjin-gu,10900,Gunja-dong,NaN,NaN,NaN,NaN,NaN,...,1000,47,NaN,2008.0,Detached / Multi-household house,22.01~24.01,New,NaN,0.0,0.0
9,2022,11230,Dongdaemun-gu,10600,Jangan-dong,1.0,Land lot,309.0,7.0,12.0,...,10000,0,NaN,2021.0,Apartment,22.01~24.01,New,NaN,0.0,NaN


The method **.shape** from the library **pandas** returns a tuple representing the dimensionality of the DataFrame. It is used to get the number of rows and columns of the dataset.

The syntax is:
df.shape

In this analysis, `df.shape` returns the number of rows and columns of the dataset loaded from the file path `'C:\\Users\\jamol\\Datasets\\Seoul_Rentals_2022_2025.csv'`.

In [5]:
df.shape

(400000, 23)

The method **.columns** from the library **pandas** returns the column labels of the DataFrame. It is used to get the names of all columns in the dataset.

The syntax is:
df.columns

In this analysis, `df.columns` returns the column labels of the dataset loaded from the file path `'C:\\Users\\jamol\\Datasets\\Seoul_Rentals_2022_2025.csv'`.

In [10]:
df.columns

Index(['registration_year', 'district_code', 'district_name',
       'legal_dong_code', 'legal_dong_name', 'lot_type_code', 'lot_type',
       'main_lot_number', 'sub_lot_number', 'floor', 'contract_date',
       'lease_type', 'leased_area_sqm', 'deposit_10k_krw',
       'monthly_rent_10k_krw', 'building_name', 'year_built', 'building_type',
       'contract_period', 'contract_type', 'renewal_right_used',
       'previous_deposit_10k_krw', 'previous_monthly_rent_10k_krw'],
      dtype='str')

The method **.info()** from the library **pandas** returns a concise summary of the DataFrame. It is used to get information about the DataFrame, including the number of rows, columns, non-null values, and data types.

The syntax is:
df.info()

In this analysis, `df.info()` returns a concise summary of the dataset loaded from the file path `'C:\\Users\\jamol\\Datasets\\Seoul_Rentals_2022_2025.csv'`.

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 400000 entries, 0 to 399999
Data columns (total 23 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   registration_year              400000 non-null  int64  
 1   district_code                  400000 non-null  int64  
 2   district_name                  400000 non-null  str    
 3   legal_dong_code                400000 non-null  int64  
 4   legal_dong_name                399988 non-null  str    
 5   lot_type_code                  299849 non-null  float64
 6   lot_type                       299849 non-null  str    
 7   main_lot_number                299976 non-null  float64
 8   sub_lot_number                 299976 non-null  float64
 9   floor                          299870 non-null  float64
 10  contract_date                  400000 non-null  str    
 11  lease_type                     400000 non-null  str    
 12  leased_area_sqm                400000 non

The method **.value__counts()** from the library **pandas** returns a Series containing counts of unique values. It is used to get the frequency of each unique value in a column of the DataFrame.

The syntax is:
df['column_name'].value_counts()

- `df['lease_type'].value_counts()` returns the frequency of each unique value in the `lease_type` column.
- `df['registration_year'].value_counts()` returns the frequency of each unique value in the `registration_year` column.
- `df['building_type'].value_counts()` returns the frequency of each unique value in the `building_type` column.

In [6]:
df['lease_type'].value_counts()

lease_type
Monthly rent             219690
Jeonse (deposit only)    180310
Name: count, dtype: int64

In [7]:
df['registration_year'].value_counts()

registration_year
2023    100028
2024    100000
2025    100000
2022     99963
2021         9
Name: count, dtype: int64

In [8]:
df['building_type'].value_counts()

building_type
Apartment                           162006
Detached / Multi-household house    100130
Row house / Villa (multi-unit)       86481
Officetel                            51383
Name: count, dtype: int64